# Basic RAG

This notebook demonstrates a basic Retrieval-Augmented Generation (RAG) flow for the GeForce NOW FAQ data. The flow covers creating an Elasticsearch text/vector index, searching the index for relevant FAQ entries, and using the search results to augment a language model's responses.

### Initialize clients

In [1]:
from elasticsearch import Elasticsearch
from constants import DEFAULT_ES_URL, DEFAULT_INDEX, DEFAULT_VECTOR_INDEX

client = Elasticsearch(DEFAULT_ES_URL)
if not client.ping():
    raise RuntimeError(f"Cannot connect to Elasticsearch at {DEFAULT_ES_URL}")

In [2]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

## 1. RAG with text search

### Create text index

Create an Elasticsearch text index for the GeForce NOW FAQ data. Searchable fields are:
- question
- answer
- tag

In [ ]:
from pathlib import Path
from faq_text_index import create_index, index_csv_data

data_path = Path().resolve().parent / "data" / "csv" / "geforce_now_faq.csv"

create_index(client, DEFAULT_INDEX, recreate=True)
inserted = index_csv_data(client, DEFAULT_INDEX, data_path)
print(f"Indexed {inserted} FAQ documents into '{DEFAULT_INDEX}' index.")

Indexed 98 FAQ documents into 'geforce-now-faq' index.


In [20]:
# Test index was created
client.get(index=DEFAULT_INDEX, id="1")
# client.search(index=DEFAULT_INDEX, query={"match": {"tag": "What is Geforce Now?"}})

ObjectApiResponse({'_index': 'geforce-now-faq', '_id': '1', '_version': 1, '_seq_no': 0, '_primary_term': 1, 'found': True, '_source': {'id': '1', 'category': 'General Questions', 'tag': 'what-is-geforce-now', 'question': 'What is GeForce NOW?', 'answer': 'GeForce NOW is NVIDIA’s cloud- game streaming service, delivering real-time RTX-powered gameplay straight from the cloud to your laptop, desktop, Mac, Chromebook, SHIELD TV, select Samsung and LG TVs, iPhone, iPad, Android devices, Steam Deck, VR headsets, Linux PC and more. Connect to your favorite game store accounts and stream games you own, or check out hundreds of favorite free-to-play games. With cloud saves for supported games, you can pick up your game where you left off, on any supported device, wherever you are.'}})

### Text Search example

Search the text index for most relevant documents based on the given query/question.
By default no `boost_dict` is applied, meaning all fields have equal weight.

In [16]:
import faq_text_search as text_search

# query = "What is Geforce Now?"
query = "How can I run games in 4K resolution?"
# boost_dict = {"question": 1, "tag": 1, "answer": 3}
results = text_search.search_faq(client, DEFAULT_INDEX, query)

for res in results:
    print(res["_source"])

{'id': '70', 'category': 'Install-to-Play', 'tag': 'storage-location-change', 'question': 'If I’m in a data center in one region and install games with permanent storage, then switch to a data center in another region, will that affect the games I play on Install-to-Play?', 'answer': 'When you first purchase an Install-to-Play game, your persistent storage location is set. You can change it later if needed, but keep in mind — changing it will erase all of your saved data for Install-to-Play games. Also, note that for Install-to-Play titles, the server location setting in the GeForce NOW app won’t apply; these games always run from the storage location you selected. The server location preference still works normally for Ready-to-Play games.'}
{'id': '76', 'category': 'PC/MAC', 'tag': 'launch-game', 'question': 'How do I launch and play a game?', 'answer': 'First, use the search bar to find the games you own and add them to your GeForce NOW Library. Once added, you can click on the game

### RAG pipeline with text search

Now we run the full pipeline using `RAG` class and a text search function. The pipeline covers:
- retrieval - get most relevant documents from Elasticseatch index using text search function
- augmentation - build a prompt using general instructions, a context (retrieved documents) and a user query
- generation - send a prompt to LLM and receive a **grounded** anwser

In [5]:
from rag import RAG, DEFAULT_INSTRUCTIONS

print(DEFAULT_INSTRUCTIONS)

You answer questions about the GeForce NOW service.
Use only the information in the provided context to answer the user's query.
If the answer cannot be found in the context, answer exactly: I don't know.
Keep the answer concise and factual.


In [17]:
assistant = RAG(
    index_name=DEFAULT_INDEX,
    search_function=text_search.search_faq,
    llm_client=openai_client,
    instructions=DEFAULT_INSTRUCTIONS,
    model_name="gpt-5.4-mini")

search_kwargs = {
    "client": client
}

records = assistant.retrieve(query, **search_kwargs)
prompt = assistant.augment(query, records)
print(prompt[1]["content"])

Context:
1. Category: Install-to-Play
   Question: If I’m in a data center in one region and install games with permanent storage, then switch to a data center in another region, will that affect the games I play on Install-to-Play?
   Answer: When you first purchase an Install-to-Play game, your persistent storage location is set. You can change it later if needed, but keep in mind — changing it will erase all of your saved data for Install-to-Play games. Also, note that for Install-to-Play titles, the server location setting in the GeForce NOW app won’t apply; these games always run from the storage location you selected. The server location preference still works normally for Ready-to-Play games.

2. Category: PC/MAC
   Question: How do I launch and play a game?
   Answer: First, use the search bar to find the games you own and add them to your GeForce NOW Library. Once added, you can click on the game tile and launch the game on your GeForce NOW gaming rig in the cloud. It will loo

In [18]:
print(assistant.generate(prompt))

GeForce NOW recommends a 45 Mbps connection for streaming up to 4K at 120 FPS. A hardwired Ethernet connection or a 5GHz wireless router is also recommended.


In [15]:
# Use a single RAG method

print(assistant.run(query, **search_kwargs))

GeForce NOW recommends at least 45 Mbps for streaming up to 4K at 120 FPS, and a hardwired Ethernet connection or a 5GHz wireless router.


## 2. RAG with Vector Search

### Create vector index

Create an Elasticsearch vector index for the GeForce NOW FAQ data. Embedded fields are:
- question
- answer
- tag
- composite of question and anwser

In [12]:
from faq_vector_index import create_default_index, index_csv_data

create_default_index(client, DEFAULT_VECTOR_INDEX, recreate=True)
inserted = index_csv_data(client, DEFAULT_VECTOR_INDEX, data_path)
print(f"Indexed {inserted} FAQ documents into '{DEFAULT_VECTOR_INDEX}' index.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Indexed 98 FAQ documents into 'geforce-now-faq-vector' index.


In [ ]:
# Test index was created - print sample vector
print("tag vector: ",client.get(index=DEFAULT_VECTOR_INDEX, id="1", source_exclude_vectors=False)["_source"]["tag_vector"])

tag vector:  [0.0017171071, -0.02689127, -0.041852966, -0.04411118, 0.092377506, 0.026560117, -0.05805113, 0.032222852, -0.056199636, 0.011924136, 0.026898017, 0.07190288, -0.05650154, 0.026137244, -0.014732136, 0.03996515, 0.07793926, -0.076527245, 0.0247828, 0.037321147, -0.0548739, -0.0056111887, -0.04196675, -0.049429785, -0.004877606, 0.049769387, -0.02321078, -0.032902606, 0.01019602, -0.0023096548, -0.014390569, 0.048062634, 0.005551403, 0.07223071, -0.1167429, -0.03395364, 0.047216907, -0.06526339, 0.00015210481, -0.036127664, -0.077372745, -0.049746364, -0.02653348, 0.015032273, 0.07157522, 0.037698228, 0.059446063, -0.045891233, 0.06482764, 0.008826508, 0.0567473, -0.016949821, 0.015741369, -0.08827585, -0.0043517635, 0.053470932, 0.037409246, -0.10795641, -0.0058718254, 0.015475467, 0.066095576, -0.059477266, -0.0004963292, 0.027552148, 0.038742904, -0.04133089, 0.031131439, -0.0376213, 0.039318386, -0.024429038, 0.003185418, 0.0027106763, -0.019536993, -0.07077862, -0.06700

### Vector Search example

Search the vector index for most relevant documents based on the given query/question. By default a composed question/anwser vector `question_answer_vector` is compared. Query is encoded (converted to vector) on the run.

In [21]:
import faq_vector_search as vector_search

query = "How can I run games in 4K resolution?"
results = vector_search.search_faq(client, DEFAULT_VECTOR_INDEX, query, size=6)

for res in results:
    print(res["_source"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{'id': '9', 'category': 'General Questions', 'tag': 'system-requirements', 'question': 'What are the minimum requirements, including internet speed, to play GeForce NOW?', 'answer': 'Please visit the System Requirements page for info on system requirements, including internet speed.'}
{'id': '39', 'category': 'Ultimate Memberships', 'tag': 'system-requirements', 'question': 'What internet or system requirements are recommended for GeForce NOW Ultimate memberships?', 'answer': 'GeForce NOW recommends at least 65Mbps for streaming up to 5120x2180p at 120 fps, 55Mbps for 2560x1440p at 240 FPS for most iMacs or 2560x1600p at 240 FPS for most MacBooks, 48Mbps for 1920x1080 at 360 FPS, 45Mbps for streaming up to 4K at 120 FPS, and 25 Mbps for FHD resolutions at 60 FPS. And for the new Cinematic Quality Streaming mode, we recommend 100 Mbps for the best quality. We also recommend a hardwired Ethernet connection, or a 5GHz wireless router. Please visit the System Requirements page for specific

### RAG pipeline with vector search

Now we run the full pipeline using `RAG` class and a vector search function. The pipeline covers:
- retrieval - get most relevant documents from Elasticseatch index using vector search function
- augmentation - build a prompt using general instructions, a context (retrieved documents) and a user query
- generation - send a prompt to LLM and receive a **grounded** anwser

In [22]:
assistant = RAG(
    index_name=DEFAULT_VECTOR_INDEX,
    search_function=vector_search.search_faq,
    llm_client=openai_client,
    instructions=DEFAULT_INSTRUCTIONS,
    model_name="gpt-5.4-mini")

search_kwargs = {
    "client": client
}

records = assistant.retrieve(query, **search_kwargs)
prompt = assistant.augment(query, records)
print(prompt[1]["content"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Context:
1. Category: General Questions
   Question: What are the minimum requirements, including internet speed, to play GeForce NOW?
   Answer: Please visit the System Requirements page for info on system requirements, including internet speed.

2. Category: Ultimate Memberships
   Question: What internet or system requirements are recommended for GeForce NOW Ultimate memberships?
   Answer: GeForce NOW recommends at least 65Mbps for streaming up to 5120x2180p at 120 fps, 55Mbps for 2560x1440p at 240 FPS for most iMacs or 2560x1600p at 240 FPS for most MacBooks, 48Mbps for 1920x1080 at 360 FPS, 45Mbps for streaming up to 4K at 120 FPS, and 25 Mbps for FHD resolutions at 60 FPS. And for the new Cinematic Quality Streaming mode, we recommend 100 Mbps for the best quality. We also recommend a hardwired Ethernet connection, or a 5GHz wireless router. Please visit the System Requirements page for specific device compatibility, hardware recommendations and details on supported regions. You

In [23]:
print(assistant.generate(prompt))

GeForce NOW recommends at least 45 Mbps for streaming up to 4K at 120 FPS. For the best experience, it also recommends a hardwired Ethernet connection or a 5GHz wireless router.


In [24]:
# Use a single RAG method

print(assistant.run(query, **search_kwargs))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Please visit the System Requirements page for specific device compatibility, hardware recommendations and details on supported regions. GeForce NOW recommends 45 Mbps for streaming up to 4K at 120 FPS.
